# 00. 데이터 정제와 동 단위 특징표

**목적:** 01~06 재분석과 이후 ml-engineer(07)가 함께 쓰는 정제 거래 자료와 동 단위 특징표를 원자료에서 새로 만든다.

**방법과 기준**
- 원자료 `data/서울_아파트_매매_실거래가_2017-2026_통합.xlsx`(시트 '아파트(매매) 실거래가')를 한 번 읽어 parquet로 저장한다.
- **행을 삭제하지 않는다.** 취소·중복·일괄 거래·미완결 월·비현실 값은 모두 플래그 열로 표시하고, 분석 표본 여부(`in_base`, `in_base_nobulk`)도 열로 남긴다.
- 동 단위 특징(학교·학원·통근시간·지하철·버스·사업체·종사자·단지 규모)은 이 노트북에서만 만든다. 04·05·02b와 ml-engineer는 `dong_features.parquet` 하나만 쓴다.

**사용한 교재 장:** 6장 전처리(결측·이상치 처리). **교재에서 바꾼 점:** 교재 6장은 IQR 기준으로 이상치 행을 삭제하지만, 여기서는 삭제하지 않고 플래그로만 표시한다. 고가 거래가 실제 값인 경우가 많고, 일괄 거래·취소 거래는 통계적 이상치가 아니라 거래 성격의 문제이기 때문이다(조율안 1절 2항).

**외부 API:** 이번 실행에서는 Google Routes API를 호출하지 않았다(API 수집하지 않음). 통근시간은 저장된 `data/05_직주근접/동별_업무지구_대중교통시간_구글.csv`만 쓰고, 이 파일에 없는 동은 결측으로 두고 `commute_missing=True`로 표시한다. 구 평균 등으로 대체하지 않는다.

In [1]:
import os
import sys
from pathlib import Path


def _find_root():
    env = os.environ.get("REANALYSIS_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "AGENTS.md").exists():
            return p
    raise FileNotFoundError("AGENTS.md가 있는 프로젝트 루트를 찾지 못했습니다")


ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src" / "재검증"))
import re
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import reanalysis_utils as ru

SEED = ru.SEED
DATA = ROOT / "data"
OUT = ru.out_dir(ROOT, "00_정제")
RAW_XLSX = DATA / "서울_아파트_매매_실거래가_2017-2026_통합.xlsx"

## 1. 원자료 읽기
모든 열을 문자열로 읽고, 원래 행 순서를 `row_id`(0부터)로 고정한다. `NO`는 9개 다운로드 묶음마다 1부터 다시 시작하므로 고유키가 아니다.

In [2]:
raw = pd.read_excel(RAW_XLSX, sheet_name="아파트(매매) 실거래가", dtype=str)
raw = raw.reset_index(drop=True)
N_RAW = len(raw)
pd.DataFrame({"rows": [N_RAW], "cols": [raw.shape[1]]})

,rows,cols
0,547735,21


원자료는 547,735행(헤더 제외), 21열이다. 이 행 수를 정제 결과에서도 그대로 유지한다.

In [3]:
tx = pd.DataFrame({"row_id": np.arange(N_RAW, dtype=np.int64)})
tx["src_no"] = raw["NO"].astype(str)
parts = raw["시군구"].astype(str).str.split()
tx["sido"] = parts.str[0]
tx["gu"] = parts.str[1]
tx["dong"] = parts.str[2]
tx["dong_key_missing"] = tx["gu"].isna() | tx["dong"].isna() | (parts.str.len() != 3)
tx["dong_id"] = (tx["gu"] + " " + tx["dong"]).where(~tx["dong_key_missing"])
tx["complex_name"] = raw["단지명"].astype(str)
tx["jibun"] = raw["번지"].astype(str)
tx["road_name"] = raw["도로명"].astype(str)
tx["complex_id"] = tx["gu"] + "|" + tx["dong"] + "|" + tx["complex_name"] + "|" + tx["jibun"]
tx["area_m2"] = pd.to_numeric(raw["전용면적(㎡)"], errors="coerce")
tx["floor"] = pd.to_numeric(raw["층"], errors="coerce")
tx["build_year"] = pd.to_numeric(raw["건축년도"], errors="coerce")
tx["price_manwon"] = pd.to_numeric(raw["거래금액(만원)"].str.replace(",", "", regex=False), errors="coerce")
tx["contract_date"] = pd.to_datetime(raw["계약년월"] + raw["계약일"].str.zfill(2), format="%Y%m%d", errors="coerce")
tx["contract_ym"] = tx["contract_date"].dt.strftime("%Y-%m")
tx["contract_year"] = tx["contract_date"].dt.year
tx["price_eok"] = tx["price_manwon"] / 1e4
tx["ppa_manwon"] = tx["price_manwon"] / tx["area_m2"]
tx["log_ppa"] = np.log(tx["ppa_manwon"])
tx["age"] = tx["contract_year"] - tx["build_year"]
tx["deal_type"] = raw["거래유형"].replace("-", pd.NA)
tx["cancel_date"] = pd.to_datetime(raw["해제사유발생일"].replace("-", pd.NA), format="%Y%m%d", errors="coerce")
tx["registration_date"] = raw["등기일자"].replace("-", pd.NA)
parse_fail = pd.Series({
    "dong_key_missing": int(tx["dong_key_missing"].sum()),
    "price_na": int(tx["price_manwon"].isna().sum()),
    "area_na": int(tx["area_m2"].isna().sum()),
    "date_na": int(tx["contract_date"].isna().sum()),
    "floor_na": int(tx["floor"].isna().sum()),
    "build_year_na": int(tx["build_year"].isna().sum()),
    "n_gu": tx["gu"].nunique(), "n_dong_id": tx["dong_id"].nunique(),
    "date_min": str(tx["contract_date"].min().date()), "date_max": str(tx["contract_date"].max().date()),
})
parse_fail.to_frame("value")

,value
dong_key_missing,0
price_na,0
area_na,0
date_na,0
floor_na,0
build_year_na,0
n_gu,25
n_dong_id,338
date_min,2017-10-28
date_max,2026-10-26


가격·면적·날짜·층·건축년도의 파싱 실패는 없고, 구는 25개, (구, 법정동) 조합은 338개다. 계약일은 2017-10-28부터 2026-10-26까지다. 따라서 **2017-10과 2026-10은 일부 날짜만 있는 부분 월**이다.

## 2. 플래그 규칙
### 2-1. 취소(해제) 거래와 취소 정보 공백
`해제사유발생일`이 있으면 취소 거래다. 계약월별 취소 표시 비율을 보면 2020-02 이전 계약에는 표시가 전혀 없다(아래 표). 이 기간은 취소 거래를 걸러낼 수 없으므로 `cancel_unobserved=True`로 표시한다.

In [4]:
tx["is_cancelled"] = tx["cancel_date"].notna()
cr = tx.groupby("contract_ym")["is_cancelled"].mean()
cr.loc["2019-10":"2020-06"].rename("cancel_share").to_frame().T

contract_ym,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06
cancel_share,0.0,0.0,0.0,0.0,0.014043,0.046828,0.043286,0.04365,0.048464


In [5]:
tx["cancel_unobserved"] = tx["contract_ym"] < "2020-02"
lag_days = (tx.loc[tx["is_cancelled"], "cancel_date"] - tx.loc[tx["is_cancelled"], "contract_date"]).dt.days
pd.Series({"cancel_rows": int(tx["is_cancelled"].sum()),
           "first_cancel_date": str(tx["cancel_date"].min().date()),
           "cancel_lag_le30d_share": float((lag_days <= 30).mean()),
           "cancel_lag_le90d_share": float((lag_days <= 90).mean())}).to_frame("value")

,value
cancel_rows,17606
first_cancel_date,2020-02-21
cancel_lag_le30d_share,0.455981
cancel_lag_le90d_share,0.829206


취소 신고는 계약 후 30일 안에 약 절반, 90일 안에 대부분이 들어온다. 자료 내려받은 시점(2026-10 말) 기준으로 최근 2~3개월 계약은 앞으로 취소될 거래가 아직 남아 있을 수 있다. 그래서 다음 두 가지를 구분한다.
- `is_partial_month`: 2017-10(10-28부터), 2026-10(10-26까지). 날짜 일부만 있는 달이다. 기본 표본에서 뺀다.
- `is_recent_provisional`: 2026-08, 2026-09. 취소·신고 지연이 끝나지 않은 잠정 월이다. 표본에는 남기고 결과에서 잠정으로 표시한다.

### 2-2. 완전 중복
`NO`를 뺀 20개 열이 모두 같은 행은 같은 거래가 두 번 실린 것으로 본다. 첫 행은 남기고 두 번째부터 `is_dup_exact=True`로 표시한다. 등기일자만 다른 쌍처럼 일부 열만 같은 경우는 별도 세대 거래일 수 있으므로 중복으로 보지 않는다.

In [6]:
dup_cols = [c for c in raw.columns if c != "NO"]
tx["is_dup_exact"] = raw.duplicated(subset=dup_cols, keep="first").to_numpy()
pd.Series({"dup_exact_rows": int(tx["is_dup_exact"].sum()),
           "dup_exact_among_noncancel": int((tx["is_dup_exact"] & ~tx["is_cancelled"]).sum())}).to_frame("value")

,value
dup_exact_rows,4506
dup_exact_among_noncancel,4180


### 2-3. 일괄 거래(같은 단지·같은 날 다수 거래)
단지 키(구+법정동+단지명+번지)와 계약일이 같은 유효 거래(취소·중복 제외) 수를 `bulk_n`으로 저장한다. 아래 표는 단지-일 묶음의 크기별 빈도와, 크기가 1 늘 때 빈도가 줄어드는 비율이다.

In [7]:
valid = ~tx["is_cancelled"] & ~tx["is_dup_exact"]
cd_key = tx["complex_id"] + "#" + tx["contract_date"].dt.strftime("%Y%m%d")
grp_n = cd_key[valid].value_counts()
tx["bulk_n"] = cd_key.map(grp_n).fillna(0).astype(int)
size_freq = grp_n.value_counts().sort_index()
tbl = pd.DataFrame({"group_size": size_freq.index, "n_groups": size_freq.values})
tbl["ratio_next"] = tbl["n_groups"].shift(-1) / tbl["n_groups"]
tbl.head(16)

,group_size,n_groups,ratio_next
0,1,417823,0.089703
1,2,37480,0.169184
2,3,6341,0.235294
3,4,1492,0.270107
4,5,403,0.441687
5,6,178,0.483146
6,7,86,0.395349
7,8,34,0.617647
8,9,21,0.714286
9,10,15,0.866667


묶음 크기 1~7에서는 크기가 1 늘 때마다 빈도가 절반 아래로 줄어든다(위 표의 `ratio_next` 0.09~0.48). 일반 거래가 우연히 같은 날 겹치는 경우에 기대되는 빠른 감소다. 크기 8부터는 감소비가 0.5를 넘고 1 이상인 구간도 나오며 꼬리가 평평해진다. 즉 이 구간은 우연한 겹침이 아니라 다른 성격의 거래(통매각·일괄 분양형 거래 등)로 볼 수 있다. **기준은 N=10(같은 단지·같은 날 유효 거래 10건 이상)으로 고정한다.** 평평한 꼬리가 시작되는 8~9 바로 뒤의 정수값이다. 다른 기준으로 민감도를 보려면 `bulk_n` 열을 그대로 쓰면 된다.

In [8]:
BULK_N = 10
tx["is_bulk"] = valid & (tx["bulk_n"] >= BULK_N)
b = tx[tx["is_bulk"]]
pd.Series({"bulk_rows": len(b), "bulk_groups": int(grp_n[grp_n >= BULK_N].size),
           "bulk_area_median_m2": float(b["area_m2"].median()),
           "bulk_share_area_lt30": float((b["area_m2"] < 30).mean()),
           "nonbulk_share_area_lt30": float((tx.loc[valid & ~tx["is_bulk"], "area_m2"] < 30).mean()),
           "bulk_rows_2026": int((b["contract_year"] == 2026).sum())}).to_frame("value")

,value
bulk_rows,4029.000000
bulk_groups,147.000000
bulk_area_median_m2,28.300000
bulk_share_area_lt30,0.555225
nonbulk_share_area_lt30,0.032928
bulk_rows_2026,670.000000


일괄 거래는 대부분 30㎡ 미만 소형이고(일반 거래는 소수), 2026년에 많이 몰려 있다. 01은 이 거래를 포함한 결과와 뺀 결과를 나란히 보이고, 02~06의 횡단면·패널 분석은 뺀 표본(`in_base_nobulk`)을 주결과로 쓴다.

### 2-4. 비현실 값·주의 값(삭제하지 않고 표시만)
- `is_floor_nonpositive`: 층 ≤ 0(지하층 표기). 실제 거래일 수 있다.
- `is_age_negative`: 계약연도 < 건축년도. 준공 전 계약이 등록된 경우로 보인다. 연식 구간은 0~4년에 넣는다.
- `is_small_unit`: 전용 20㎡ 미만. 도시형·초소형 단지가 많다.
- `is_price_outlier`: 같은 구·같은 연도 안에서 log(㎡당 가격)의 로버스트 z(중위수·MAD 기준)가 5를 넘는 거래. 규칙만 미리 정했고 표본에서 빼지 않는다.
- `is_direct_trade`: 거래유형이 직거래. 거래유형은 2021-11 이후만 기록되어 있다.

In [9]:
tx["is_floor_nonpositive"] = tx["floor"] <= 0
tx["is_age_negative"] = tx["age"] < 0
tx["is_small_unit"] = tx["area_m2"] < 20
g = tx.groupby(["gu", "contract_year"])["log_ppa"]
med = g.transform("median")
mad = g.transform(lambda s: (s - s.median()).abs().median())
tx["price_robust_z"] = (tx["log_ppa"] - med) / (1.4826 * mad)
tx["is_price_outlier"] = tx["price_robust_z"].abs() > 5
tx["is_direct_trade"] = tx["deal_type"].eq("직거래").fillna(False).astype(bool)
first_deal_type = tx.loc[tx["deal_type"].notna(), "contract_ym"].min()
pd.Series({"floor_min": tx["floor"].min(), "first_month_with_deal_type": first_deal_type}).to_frame("value")

,value
floor_min,-3
first_month_with_deal_type,2021-11


### 2-5. 미완결 월과 분석 표본
- `is_partial_month`: 2017-10, 2026-10
- `is_recent_provisional`: 2026-08, 2026-09
- `in_base`: 취소 아님 & 완전 중복 아님 & 부분 월 아님 & 동 키 있음. 기간은 2017-11~2026-09다.
- `in_base_nobulk`: `in_base` & 일괄 거래 아님

In [10]:
tx["is_partial_month"] = tx["contract_ym"].isin(["2017-10", "2026-10"])
tx["is_recent_provisional"] = tx["contract_ym"].isin(["2026-08", "2026-09"])
tx["in_base"] = (~tx["is_cancelled"] & ~tx["is_dup_exact"] & ~tx["is_partial_month"] & ~tx["dong_key_missing"])
tx["in_base_nobulk"] = tx["in_base"] & ~tx["is_bulk"]

### 2-6. 분석용 구간 변수
03·04·05·06이 같은 구간을 쓰도록 여기서 정한다. 회귀의 기준 범주는 면적 `060-085`, 연식 `10-14`, 층 `06-10`이다.
- 면적(㎡): ≤40, 40~60, 60~85, 85~102, 102~135, 135 초과
- 연식(계약연도−건축년도): 0~4(음수 포함), 5~9, …, 30~34, 35 이상
- 층: 지하(≤0), 1, 2, 3~5, 6~10, 11~15, 16~20, 21~30, 31 이상

In [11]:
tx["area_bin"] = pd.cut(tx["area_m2"], [0, 40, 60, 85, 102, 135, np.inf],
                        labels=["000-040", "040-060", "060-085", "085-102", "102-135", "135+"]).astype(str)
age_c = tx["age"].clip(lower=0)
tx["age_bin"] = pd.cut(age_c, [-1, 4, 9, 14, 19, 24, 29, 34, np.inf],
                       labels=["00-04", "05-09", "10-14", "15-19", "20-24", "25-29", "30-34", "35+"]).astype(str)
tx["floor_bin"] = pd.cut(tx["floor"], [-np.inf, 0, 1, 2, 5, 10, 15, 20, 30, np.inf],
                         labels=["B", "01", "02", "03-05", "06-10", "11-15", "16-20", "21-30", "31+"]).astype(str)
tx[["area_bin", "age_bin", "floor_bin"]].apply(lambda s: s.value_counts()).fillna(0).astype(int).T

,00-04,000-040,01,02,03-05,040-060,05-09,06-10,060-085,085-102,...,135+,15-19,16-20,20-24,21-30,25-29,30-34,31+,35+,B
area_bin,0,45677,0,0,0,183159,0,0,225947,13924,...,21492,0,0,0,0,0,0,0,0,0
age_bin,46837,0,0,0,0,0,59615,0,0,0,...,0,95486,0,98740,0,73918,56941,0,50015,0
floor_bin,0,0,28344,34976,112036,0,0,165036,0,0,...,0,0,51525,0,25422,0,0,3898,0,120


## 3. 플래그별 행 수

In [12]:
flag_cols = ["dong_key_missing", "is_cancelled", "cancel_unobserved", "is_dup_exact", "is_bulk",
             "is_partial_month", "is_recent_provisional", "is_floor_nonpositive", "is_age_negative",
             "is_small_unit", "is_price_outlier", "is_direct_trade", "in_base", "in_base_nobulk"]
flag_counts = pd.DataFrame({"flag": flag_cols,
                            "rows": [int(tx[c].sum()) for c in flag_cols]})
flag_counts["share_pct"] = flag_counts["rows"] / N_RAW * 100
flag_counts.to_csv(OUT / "flag_counts.csv", index=False, encoding="utf-8-sig")
flag_counts

,flag,rows,share_pct
0,dong_key_missing,0,0.000000
1,is_cancelled,17606,3.214328
2,cancel_unobserved,180866,33.020713
3,is_dup_exact,4506,0.822661
4,is_bulk,4029,0.735575
5,is_partial_month,2915,0.532192
6,is_recent_provisional,7185,1.311766
7,is_floor_nonpositive,120,0.021908
8,is_age_negative,40,0.007303
9,is_small_unit,15338,2.800259


`in_base`(기본 표본)와 `in_base_nobulk`(일괄 거래 제외 표본)의 행 수가 01~06의 분석 표본 크기다. 2020-02 이전 계약(`cancel_unobserved`)은 취소 거래가 섞여 있을 수 있어, 02·06에서 이 구간을 쓸 때 한계로 적는다.

## 4. 저장(transactions.parquet)

In [13]:
assert len(tx) == N_RAW, "행 수가 원자료와 다릅니다"
tx = tx.drop(columns=["sido"])
tx.to_parquet(OUT / "transactions.parquet", index=False)
pd.Series({"rows_saved": len(tx), "cols_saved": tx.shape[1]}).to_frame("value")

,value
rows_saved,547735
cols_saved,42


## 5. 동 단위 특징표(dong_features.parquet)
기본키는 (`gu`, `dong`), 조인 키는 `dong_id`="<구> <법정동>"이다. 동 이름만으로는 겹친다(예: 강남구·은평구 신사동). 동 목록은 거래 자료에 나오는 338개 (구, 법정동)이다.

### 5-1. 동 대표 좌표
- 1순위: 서울시 공동주택 아파트 정보(`아파트 좌표정보_…csv`)의 단지 좌표 중앙값. 이 파일의 (시군구, 읍면동)을 그대로 쓰고, 행정동 표기(예: 개봉3동)는 숫자를 뗀 법정동 이름(개봉동)이 목록에 있을 때만 그쪽으로 붙인다.
- 2순위: 단지 좌표가 없는 동은 같은 법정동에 있는 초중고 좌표의 중앙값(`coord_source="school_median"`).
- 둘 다 없으면 좌표 결측(`coord_missing=True`).

In [14]:
dongs = (tx.loc[~tx["dong_key_missing"], ["gu", "dong", "dong_id"]].drop_duplicates()
         .sort_values(["gu", "dong"]).reset_index(drop=True))
DONG_SET = set(zip(dongs["gu"], dongs["dong"]))


def norm_dong(gu, dong):
    """(구, 동)이 목록에 없으면 행정동 숫자를 뗀 법정동 이름으로 바꿔 본다."""
    if (gu, dong) in DONG_SET:
        return dong
    alt = re.sub(r"\d+동$", "동", str(dong))
    return alt if (gu, alt) in DONG_SET else dong


apt = ru.read_csv_any(DATA / "05_직주근접" / "아파트 좌표정보_서울시 공동주택 아파트 정보.csv")
apt = apt.rename(columns={"주소(시군구)": "gu", "주소(읍면동)": "dong_raw", "좌표X": "lon", "좌표Y": "lat",
                          "k-전체세대수": "households", "k-단지분류(아파트,주상복합등등)": "complex_type"})
apt["dong"] = [norm_dong(g, d) for g, d in zip(apt["gu"], apt["dong_raw"])]
apt["households"] = apt["households"].where(apt["households"] > 0)  # 0세대(70개 단지, 예: 대단지 주상복합)는 기록 누락으로 보고 결측 처리
apt["in_list"] = [(g, d) in DONG_SET for g, d in zip(apt["gu"], apt["dong"])]
apt["coord_ok"] = apt["lat"].between(37.4, 37.75) & apt["lon"].between(126.7, 127.25)
pd.Series({"apt_rows": len(apt), "apt_coord_ok": int(apt["coord_ok"].sum()),
           "apt_in_dong_list": int(apt["in_list"].sum()),
           "renamed_admin_dong": int((apt["dong"] != apt["dong_raw"]).sum()),
           "dongs_with_apt_coord": apt.loc[apt["coord_ok"] & apt["in_list"], ["gu", "dong"]].drop_duplicates().shape[0]}
          ).to_frame("value")

,value
apt_rows,2898
apt_coord_ok,2858
apt_in_dong_list,2898
renamed_admin_dong,3
dongs_with_apt_coord,276


In [15]:
school = ru.read_csv_any(DATA / "04_교육" / "전국초중등학교위치표준데이터.csv", dtype=str)
school = school[school["소재지지번주소"].str.startswith("서울특별시", na=False)].copy()
sp_ = school["소재지지번주소"].str.split()
school["gu"] = sp_.str[1]
school["dong"] = [norm_dong(g, d) for g, d in zip(school["gu"], sp_.str[2])]
school["lat"] = pd.to_numeric(school["위도"], errors="coerce")
school["lon"] = pd.to_numeric(school["경도"], errors="coerce")
school["in_list"] = [(g, d) in DONG_SET for g, d in zip(school["gu"], school["dong"])]
SCHOOL_ASOF = school["데이터기준일자"].max()

a_ok = apt[apt["coord_ok"] & apt["in_list"]]
coord_apt = a_ok.groupby(["gu", "dong"]).agg(lat=("lat", "median"), lon=("lon", "median"),
                                             coord_n_points=("lat", "size")).reset_index()
coord_apt["coord_source"] = "apt_complex_median"
s_ok = school[school["in_list"] & school["lat"].notna()]
coord_sch = s_ok.groupby(["gu", "dong"]).agg(lat=("lat", "median"), lon=("lon", "median"),
                                             coord_n_points=("lat", "size")).reset_index()
coord_sch["coord_source"] = "school_median"
coord_sch = coord_sch.merge(coord_apt[["gu", "dong"]], how="left", indicator=True)
coord_sch = coord_sch[coord_sch["_merge"] == "left_only"].drop(columns="_merge")
coords = pd.concat([coord_apt, coord_sch], ignore_index=True)
feat = dongs.merge(coords, on=["gu", "dong"], how="left", validate="one_to_one")
feat["coord_missing"] = feat["lat"].isna()
feat["coord_source"] = feat["coord_source"].fillna("none")
assert len(feat) == len(dongs)
feat["coord_source"].value_counts().to_frame("dongs")

,dongs
coord_source,
apt_complex_median,276
none,41
school_median,21


### 5-2. 교육: 학교 수와 학원 수
- **학교:** `전국초중등학교위치표준데이터.csv`에서 서울 학교(운영 중)를 지번주소의 법정동으로 센다. 이 자료는 서울 전체를 담고 있으므로, 목록에 없는 동의 학교 수 0은 "실제 0"이다.
- **학원:** `학원교습소정보.csv`(서울시교육청, 모두 '개원' 상태). 도로명주소에는 법정동이 없으므로 `도로명상세주소`의 괄호 안 첫 항목(예: "(대치동, 한보종합상가)" → 대치동)을 법정동으로 쓴다. 구는 `행정구역명` 열을 쓴다.
- **학원 유형 규칙(`분야명` 기준, 미리 고정):**
  - `acad_exam`: '입시.검정 및 보습'(보통교과·진학지도 등). 04의 주변수다.
  - `acad_lang`: '국제화'(외국어 등)
  - `acad_arts`: '예능(대)'
  - `acad_other`: 그 밖의 모든 분야(기예·직업기술·종합·독서실·인문사회·정보·특수교육·기타)
  - `acad_exam_hagwon`: `acad_exam` 가운데 '학원'만(교습소 제외). 규모가 큰 기관만 센 보조 변수다.
- 동을 찾지 못한 학원은 어느 동에도 넣지 않고, 구별 미배정 비율을 기록한다. 학원이 없는 동의 값은 0이며, 배정 실패로 일부 과소집계가 있을 수 있다.

In [16]:
sch_cnt = (school[school["in_list"]].pivot_table(index=["gu", "dong"], columns="학교급구분", values="학교ID",
                                                 aggfunc="count", fill_value=0).reset_index())
sch_cnt = sch_cnt.rename(columns={"초등학교": "n_school_elem", "중학교": "n_school_middle", "고등학교": "n_school_high"})
feat = feat.merge(sch_cnt, on=["gu", "dong"], how="left", validate="one_to_one")
for c in ["n_school_elem", "n_school_middle", "n_school_high"]:
    feat[c] = feat[c].fillna(0).astype(int)
feat["n_school_total"] = feat[["n_school_elem", "n_school_middle", "n_school_high"]].sum(axis=1)
pd.Series({"seoul_schools": len(school), "schools_in_dong_list": int(school["in_list"].sum()),
           "school_asof": SCHOOL_ASOF}).to_frame("value")

,value
seoul_schools,1313
schools_in_dong_list,1292
school_asof,2026-10-01


In [17]:
acad = ru.read_csv_any(DATA / "04_교육" / "학원교습소정보.csv", dtype=str)
acad["gu"] = acad["행정구역명"].str.strip()
paren = acad["도로명상세주소"].fillna("").str.findall(r"\(([^()]*)\)")
first_tok = paren.apply(lambda xs: xs[-1].split(",")[0].strip() if xs else "")
acad["dong_raw"] = first_tok
acad["dong"] = [norm_dong(g, d) for g, d in zip(acad["gu"], acad["dong_raw"])]
acad["matched"] = [(g, d) in DONG_SET for g, d in zip(acad["gu"], acad["dong"])]
TYPE_MAP = {"입시.검정 및 보습": "acad_exam", "국제화": "acad_lang", "예능(대)": "acad_arts"}
acad["acad_type"] = acad["분야명"].map(TYPE_MAP).fillna("acad_other")
ACAD_ASOF = pd.to_datetime(acad["수정일자"], format="%Y%m%d", errors="coerce").max()
am = acad[acad["matched"]]
ac = am.pivot_table(index=["gu", "dong"], columns="acad_type", values="학원지정번호", aggfunc="count",
                    fill_value=0).reset_index()
ac2 = (am[(am["acad_type"] == "acad_exam") & (am["학원교습소명"] == "학원")]
       .groupby(["gu", "dong"]).size().rename("acad_exam_hagwon").reset_index())
feat = feat.merge(ac, on=["gu", "dong"], how="left", validate="one_to_one").merge(
    ac2, on=["gu", "dong"], how="left", validate="one_to_one")
for c in ["acad_exam", "acad_lang", "acad_arts", "acad_other", "acad_exam_hagwon"]:
    feat[c] = feat[c].fillna(0).astype(int)
feat["acad_total"] = feat[["acad_exam", "acad_lang", "acad_arts", "acad_other"]].sum(axis=1)
acad_unassigned = (acad.assign(un=~acad["matched"]).groupby("gu")["un"].mean().rename("unassigned_share"))
acad_unassigned.to_frame().to_csv(OUT / "academy_unassigned_by_gu.csv", encoding="utf-8-sig")
pd.Series({"academies": len(acad), "matched_to_dong": int(acad["matched"].sum()),
           "matched_share": float(acad["matched"].mean()), "no_paren_dong": int((acad["dong_raw"] == "").sum()),
           "max_unassigned_share_gu": float(acad_unassigned.max()),
           "acad_asof": str(ACAD_ASOF.date())}).to_frame("value")

,value
academies,25521
matched_to_dong,24434
matched_share,0.957408
no_paren_dong,525
max_unassigned_share_gu,0.379433
acad_asof,2026-10-25


학원의 동 배정률은 위 표의 `matched_share`다. 배정하지 못한 학원은 괄호 주소가 없거나, 아파트 거래가 없는 동(상업지 등)에 있는 경우다. 구별 미배정 비율은 `academy_unassigned_by_gu.csv`에 있다.

### 5-3. 통근시간(저장된 CSV, API 미수집)
`동별_업무지구_대중교통시간_구글.csv`의 `동키`("<구> <법정동>")를 `dong_id`와 바로 맞춘다. 업무지구는 광화문·강남·여의도 3곳이고 출발시각은 파일에 적힌 단일 시각이다. 최솟값과 그 업무지구도 함께 저장한다.

In [18]:
cm = ru.read_csv_any(DATA / "05_직주근접" / "동별_업무지구_대중교통시간_구글.csv")
cm_dep = sorted(cm["출발시각"].unique().tolist())
cmw = cm.pivot_table(index="동키", columns="업무지구", values="분", aggfunc="first")
cmw = cmw.rename(columns={"광화문": "commute_gwanghwamun", "강남": "commute_gangnam", "여의도": "commute_yeouido"})
cmw["commute_min"] = cmw[["commute_gwanghwamun", "commute_gangnam", "commute_yeouido"]].min(axis=1)
cmw["commute_min_dest"] = cmw[["commute_gwanghwamun", "commute_gangnam", "commute_yeouido"]].idxmin(axis=1).map(
    {"commute_gwanghwamun": "광화문", "commute_gangnam": "강남", "commute_yeouido": "여의도"})
cmw = cmw.reset_index().rename(columns={"동키": "dong_id"})
n_before = len(feat)
feat = feat.merge(cmw, on="dong_id", how="left", validate="one_to_one")
assert len(feat) == n_before
feat["commute_missing"] = feat["commute_min"].isna()
miss = feat.loc[feat["commute_missing"], ["gu", "dong", "dong_id", "coord_source"]]
miss.to_csv(OUT / "commute_missing_dongs.csv", index=False, encoding="utf-8-sig")
pd.Series({"csv_rows": len(cm), "csv_dongs": cm["동키"].nunique(), "departure_times": "; ".join(cm_dep),
           "csv_dongs_not_in_list": int((~cmw["dong_id"].isin(feat["dong_id"])).sum()),
           "dongs_with_commute": int((~feat["commute_missing"]).sum()),
           "commute_missing_dongs": int(feat["commute_missing"].sum())}).to_frame("value")

,value
csv_rows,828
csv_dongs,276
departure_times,2026-11-18T08:00:00+09:00
csv_dongs_not_in_list,0
dongs_with_commute,276
commute_missing_dongs,62


In [19]:
miss.groupby("gu").size().sort_values(ascending=False).head(10).rename("missing_dongs").to_frame().T

gu,종로구,성북구,중구,용산구,영등포구,마포구,서대문구,광진구,구로구,동작구
missing_dongs,20,12,10,7,4,2,2,1,1,1


**이번 실행에서는 Google Routes API를 호출하지 않았다.** 통근시간이 없는 동의 수는 위 표의 `commute_missing_dongs`이고 목록은 `outputs/재검증/00_정제/commute_missing_dongs.csv`에 있다. 결측은 종로·성북·중·용산 등 도심 구에 몰려 있어 무작위 결측이 아니다. 05의 결과는 통근시간이 있는 동에 대한 결과로만 읽어야 한다.

### 5-4. 지하철·버스 접근성
- 단지 좌표가 있는 동은 **단지마다** 최근접 지하철역 거리(m), 반경 1km 역 수, 반경 500m 버스정류소 수를 구한 뒤 동 안의 중앙값을 쓴다. 학교 좌표로 대표점을 잡은 동은 그 대표점에서 계산한다.
- 지하철역은 `지하철역 좌표정보.csv` 784개 역(서울 밖 역 포함, 경계 동의 실제 최근접 역을 반영). 역 이름에서 괄호를 떼고 2개 이상 노선에 나오면 환승역으로 본다.
- 버스정류소는 `버스정류소 위치정보.csv`에서 한강버스 선착장을 뺀 정류소다.

In [20]:
st = ru.read_csv_any(DATA / "05_직주근접" / "지하철역 좌표정보.csv")
st["name"] = st["역한글명칭"].str.replace(r"\(.*\)", "", regex=True).str.strip()
st["line"] = st["호선명칭"].str.replace(r"\(.*\)", "", regex=True).str.strip()
transfer_names = set(st.groupby("name")["line"].nunique().loc[lambda s: s >= 2].index)
st["is_transfer"] = st["name"].isin(transfer_names)
bus = ru.read_csv_any(DATA / "05_직주근접" / "버스정류소 위치정보.csv")
bus = bus[bus["정류소 타입"] != "한강선착장"]
S_LAT, S_LON, S_TR = st["환승역Y좌표"].to_numpy(), st["환승역X좌표"].to_numpy(), st["is_transfer"].to_numpy()
B_LAT, B_LON = bus["Y좌표"].to_numpy(), bus["X좌표"].to_numpy()


def access(lat, lon):
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    out = []
    for i in range(0, len(lat), 500):
        la, lo = lat[i:i + 500, None], lon[i:i + 500, None]
        ds = ru.haversine_m(la, lo, S_LAT[None, :], S_LON[None, :])
        db = ru.haversine_m(la, lo, B_LAT[None, :], B_LON[None, :])
        j = ds.argmin(axis=1)
        out.append(pd.DataFrame({"subway_dist_m": ds.min(axis=1), "subway_n_1km": (ds <= 1000).sum(axis=1),
                                 "nearest_is_transfer": S_TR[j], "bus_n_500m": (db <= 500).sum(axis=1)}))
    return pd.concat(out, ignore_index=True)


acc_apt = pd.concat([a_ok[["gu", "dong"]].reset_index(drop=True), access(a_ok["lat"], a_ok["lon"])], axis=1)
acc_d1 = acc_apt.groupby(["gu", "dong"]).agg(subway_dist_m=("subway_dist_m", "median"),
                                             subway_n_1km=("subway_n_1km", "median"),
                                             nearest_transfer_share=("nearest_is_transfer", "mean"),
                                             bus_n_500m=("bus_n_500m", "median")).reset_index()
fs = feat[feat["coord_source"] == "school_median"]
acc_d2 = pd.concat([fs[["gu", "dong"]].reset_index(drop=True), access(fs["lat"], fs["lon"])], axis=1)
acc_d2 = acc_d2.rename(columns={"nearest_is_transfer": "nearest_transfer_share"})
acc_d2["nearest_transfer_share"] = acc_d2["nearest_transfer_share"].astype(float)
acc_d = pd.concat([acc_d1, acc_d2], ignore_index=True)
feat = feat.merge(acc_d, on=["gu", "dong"], how="left", validate="one_to_one")
feat[["subway_dist_m", "subway_n_1km", "bus_n_500m", "nearest_transfer_share"]].describe().T[["count", "mean", "50%", "min", "max"]]

,count,mean,50%,min,max
subway_dist_m,297.0,507.501531,437.812361,48.591777,2886.532247
subway_n_1km,297.0,3.685185,3.000000,0.000000,12.000000
bus_n_500m,297.0,22.782828,21.500000,2.500000,62.000000
nearest_transfer_share,297.0,0.291208,0.000000,0.000000,1.000000


### 5-5. 사업체·종사자(구 단위)
`합본_2015-2024_사업체수.csv`·`종사자수.csv`의 행정구역코드는 행정동 8자리·구 5자리 통계 코드다. 원자료 안에 행정동 코드와 법정동 이름을 잇는 표가 없어 **법정동 단위로는 붙일 수 없다.** 그래서 구 단위(5자리 코드) 값을 동에 붙인다. 구 코드는 통계청 시군구 코드 순서(11010 종로구 … 11250 강동구)를 따랐고, 아래 표에서 구 코드별 행정동 수로 대응이 맞는지 확인한다.
- `gu_businesses_2024`: 2024년 총괄 사업체 수(항목 `to_fa_010`)
- `gu_workers_2024`: 2024년 종사자 수(19개 산업 항목 합)

In [21]:
GU_CODE = {"11010": "종로구", "11020": "중구", "11030": "용산구", "11040": "성동구", "11050": "광진구",
           "11060": "동대문구", "11070": "중랑구", "11080": "성북구", "11090": "강북구", "11100": "도봉구",
           "11110": "노원구", "11120": "은평구", "11130": "서대문구", "11140": "마포구", "11150": "양천구",
           "11160": "강서구", "11170": "구로구", "11180": "금천구", "11190": "영등포구", "11200": "동작구",
           "11210": "관악구", "11220": "서초구", "11230": "강남구", "11240": "송파구", "11250": "강동구"}
biz = ru.read_csv_any(DATA / "05_직주근접" / "합본_2015-2024_사업체수.csv", dtype=str)
wrk = ru.read_csv_any(DATA / "05_직주근접" / "합본_2015-2024_종사자수.csv", dtype=str)
for d_ in (biz, wrk):
    d_["값"] = pd.to_numeric(d_["값"], errors="coerce")
b24 = biz[(biz["연도"] == "2024") & (biz["행정구역코드"].str.len() == 5) & (biz["항목코드"] == "to_fa_010")]
w24 = wrk[(wrk["연도"] == "2024") & (wrk["행정구역코드"].str.len() == 5)]
emp = pd.DataFrame({"gu_code": list(GU_CODE)})
emp["gu"] = emp["gu_code"].map(GU_CODE)
emp["gu_businesses_2024"] = emp["gu_code"].map(b24.set_index("행정구역코드")["값"])
emp["gu_workers_2024"] = emp["gu_code"].map(w24.groupby("행정구역코드")["값"].sum())
n_admin = (biz[(biz["연도"] == "2024") & (biz["행정구역코드"].str.len() == 8)]["행정구역코드"].str[:5]
           .value_counts())
emp["n_admin_dong_2024"] = emp["gu_code"].map(n_admin)
emp.sort_values("gu_workers_2024", ascending=False).head(6)

,gu_code,gu,gu_businesses_2024,gu_workers_2024,n_admin_dong_2024
22,11230,강남구,98306,751888.0,22
21,11220,서초구,69341,491157.0,18
18,11190,영등포구,69459,437710.0,18
23,11240,송파구,69018,411605.0,27
1,11020,중구,66044,398128.0,15
15,11160,강서구,53285,305265.0,20


위 표에서 종사자 수 상위는 강남·서초·영등포·송파·중구로 업무지구가 몰린 구와 맞고, 구 코드별 행정동 수(강남 22, 송파 27, 중구 15 등)도 각 구의 행정동 수와 맞아 구 코드 대응이 맞다고 판단했다. 구 단위 값이므로 구 고정효과를 넣은 회귀에서는 식별되지 않는다.

### 5-6. 단지 규모·연식
- `apt_n_complex`, `apt_households_median`, `apt_households_total`, `apt_large_share`(1,000세대 이상 단지 비율): 공동주택 아파트 정보 파일 기준(현재 시점 목록). 전체세대수가 0으로 적힌 단지 70개(면적별 세대 현황도 모두 0인 대단지 포함)는 기록 누락으로 보고 세대수 통계에서 뺀다(단지 수에는 포함).
- `median_build_year`, `tx_n_base`: 기본 표본 거래(2017-11~2026-09)의 건축년도 중앙값과 거래 수.

In [22]:
apt_in = apt[apt["in_list"]]
size = apt_in.groupby(["gu", "dong"]).agg(apt_n_complex=("lat", "size"),
                                          apt_households_median=("households", "median"),
                                          apt_households_total=("households", lambda s: s.sum(min_count=1)),
                                          apt_large_share=("households", lambda s: float((s.dropna() >= 1000).mean()) if s.notna().any() else np.nan)
                                          ).reset_index()
base = tx[tx["in_base"]]
txs = base.groupby(["gu", "dong"]).agg(median_build_year=("build_year", "median"), tx_n_base=("row_id", "size")).reset_index()
n0 = len(feat)
feat = feat.merge(size, on=["gu", "dong"], how="left", validate="one_to_one")
feat = feat.merge(txs, on=["gu", "dong"], how="left", validate="one_to_one")
feat = feat.merge(emp[["gu", "gu_businesses_2024", "gu_workers_2024"]], on="gu", how="left", validate="many_to_one")
assert len(feat) == n0
feat["tx_n_base"] = feat["tx_n_base"].fillna(0).astype(int)
feat["apt_info_missing"] = feat["apt_n_complex"].isna()

### 5-7. 키 점검과 저장

In [23]:
checks = pd.Series({
    "rows": len(feat),
    "dup_gu_dong": int(feat.duplicated(["gu", "dong"]).sum()),
    "dup_dong_id": int(feat["dong_id"].duplicated().sum()),
    "dong_names_shared_across_gu": int(feat.groupby("dong")["gu"].nunique().gt(1).sum()),
    "coord_missing": int(feat["coord_missing"].sum()),
    "commute_missing": int(feat["commute_missing"].sum()),
    "apt_info_missing": int(feat["apt_info_missing"].sum()),
    "tx_dong_id_not_in_features": int((~tx.loc[~tx["dong_key_missing"], "dong_id"].isin(feat["dong_id"])).sum()),
})
assert checks["dup_gu_dong"] == 0 and checks["dup_dong_id"] == 0
checks.to_frame("value")

,value
rows,338
dup_gu_dong,0
dup_dong_id,0
dong_names_shared_across_gu,2
coord_missing,41
commute_missing,62
apt_info_missing,60
tx_dong_id_not_in_features,0


In [24]:
col_order = ["gu", "dong", "dong_id", "lat", "lon", "coord_source", "coord_n_points", "coord_missing",
             "n_school_elem", "n_school_middle", "n_school_high", "n_school_total",
             "acad_exam", "acad_exam_hagwon", "acad_lang", "acad_arts", "acad_other", "acad_total",
             "commute_gwanghwamun", "commute_gangnam", "commute_yeouido", "commute_min", "commute_min_dest",
             "commute_missing", "subway_dist_m", "subway_n_1km", "nearest_transfer_share", "bus_n_500m",
             "gu_businesses_2024", "gu_workers_2024", "apt_n_complex", "apt_households_median",
             "apt_households_total", "apt_large_share", "apt_info_missing", "median_build_year", "tx_n_base"]
feat = feat[col_order]
feat.to_parquet(OUT / "dong_features.parquet", index=False)
match_report = feat[["gu", "dong", "dong_id", "coord_source", "coord_missing", "commute_missing", "apt_info_missing", "tx_n_base"]]
match_report.to_csv(OUT / "dong_match_report.csv", index=False, encoding="utf-8-sig")
feat.shape

(338, 37)

### 5-8. 특징별 시점 메타데이터(dong_features_meta.csv)
`usable_in_backtest=false`인 특징은 과거 시점 값이 없는 현재 시점 자료다. 동 대표 좌표(`lat`, `lon`)도 현재 단지·학교 목록으로 계산했으므로 false로 둔다. 현재 시점 횡단면 분석과 04·05의 연관 분석에만 쓰고, ml-engineer의 07-1 주 백테스트에는 넣지 않는다.

In [25]:
DEP = cm_dep[0][:10] if cm_dep else ""
meta_rows = [
    ("lat/lon", "아파트 좌표정보_서울시 공동주택 아파트 정보.csv; 전국초중등학교위치표준데이터.csv", "2026-11-13", "2026-11-13", False,
     "현재 단지 목록(없으면 현재 학교 목록) 좌표의 중앙값이다. 과거 시점에 없던 단지가 대표점을 끌어갈 수 있어 백테스트에 쓰지 않는다(현재 시점 횡단면·지도용)"),
    ("n_school_*", "전국초중등학교위치표준데이터.csv", SCHOOL_ASOF, SCHOOL_ASOF, False, "현재 운영 중인 학교만 있다. 과거 개교·폐교 이력 없음"),
    ("acad_*", "학원교습소정보.csv", str(ACAD_ASOF.date()), str(ACAD_ASOF.date()), False,
     "등록상태가 모두 '개원'인 현재 목록. 폐원 이력 없음. 괄호 주소로 법정동 배정"),
    ("commute_*", "동별_업무지구_대중교통시간_구글.csv", DEP, DEP, False,
     "구글 대중교통 경로, 출발 " + "; ".join(cm_dep) + ". 현재 교통망 기준. API 미수집, 파일에 없는 동은 결측"),
    ("subway_dist_m/subway_n_1km/nearest_transfer_share", "지하철역 좌표정보.csv", "미상", "미상", False,
     "파일에 기준일이 없다. 현재 역 목록으로 보고, 과거 개통 전 역이 포함될 수 있다"),
    ("bus_n_500m", "버스정류소 위치정보.csv", "미상", "미상", False, "파일에 기준일이 없다. 현재 정류소 목록으로 본다"),
    ("gu_businesses_2024/gu_workers_2024", "합본_2015-2024_사업체수.csv; 합본_2015-2024_종사자수.csv", "2024-12-31", "2025-01-01", False,
     "구 단위 값. 2024년 한 해 값만 넣었다(연도별 값으로 다시 만들면 백테스트에 쓸 수 있음). 공표 시점은 원자료에 없어 기준연도 다음 날로 적었다"),
    ("apt_*", "아파트 좌표정보_서울시 공동주택 아파트 정보.csv", "2026-11-13", "2026-11-13", False,
     "현재 단지 목록(k-수정일자 최댓값 기준). 과거에 없던 단지가 포함된다"),
    ("median_build_year/tx_n_base", "서울_아파트_매매_실거래가_2017-2026_통합.xlsx", "2026-09-30", "2026-09-30", False,
     "2017-11~2026-09 전체 기본 표본으로 계산해 미래 거래 정보가 섞여 있다"),
    ("coord_missing/commute_missing/apt_info_missing", "(파생)", "2026-11-18", "2026-11-18", False, "결측 지시 변수"),
]
apt_asof = pd.to_datetime(apt["k-수정일자"], errors="coerce").max()
meta = pd.DataFrame(meta_rows, columns=["feature", "source_file", "as_of_date", "available_from", "usable_in_backtest", "note"])
meta.loc[meta["feature"].isin(["lat/lon", "apt_*"]), ["as_of_date", "available_from"]] = str(apt_asof.date())
meta.to_csv(OUT / "dong_features_meta.csv", index=False, encoding="utf-8-sig")
meta[["feature", "as_of_date", "usable_in_backtest"]]

,feature,as_of_date,usable_in_backtest
0,lat/lon,2026-11-13,False
1,n_school_*,2026-10-01,False
2,acad_*,2026-10-25,False
3,commute_*,2026-11-18,False
4,subway_dist_m/subway_n_1km/nearest_transfer_share,미상,False
5,bus_n_500m,미상,False
6,gu_businesses_2024/gu_workers_2024,2024-12-31,False
7,apt_*,2026-11-13,False
8,median_build_year/tx_n_base,2026-09-30,False
9,coord_missing/commute_missing/apt_info_missing,2026-11-18,False


## 6. 컬럼 설명 README 작성
`outputs/재검증/00_정제/README.md`에 컬럼 설명, 플래그 규칙, 인계 규칙을 적는다.

In [26]:
fc = flag_counts.set_index("flag")["rows"].to_dict()
readme = f"""# 00_정제: 정제 거래 자료와 동 단위 특징표

생성 노트북: `src/재검증/00_데이터정제.ipynb`. 원자료 행을 삭제하지 않고 플래그로 표시했다.
**이번 실행에서는 Google Routes API를 호출하지 않았다(API 수집하지 않음).** 통근시간은 저장된 CSV만 썼고, 통근시간이 없는 동은 {int(feat['commute_missing'].sum())}개다(`commute_missing_dongs.csv`).

## 파일
| 파일 | 내용 |
|---|---|
| `transactions.parquet` | 거래 {N_RAW:,}행(원자료와 같음), 1행=원자료 1행 |
| `dong_features.parquet` | 법정동 {len(feat)}행, 기본키 (`gu`, `dong`), 조인 키 `dong_id` |
| `dong_features_meta.csv` | 특징별 기준일·관측 가능 시점·백테스트 사용 가능 여부 |
| `flag_counts.csv` | 플래그별 행 수 |
| `commute_missing_dongs.csv` | 통근시간 결측 동 목록 |
| `dong_match_report.csv` | 동별 좌표·통근·단지정보 매칭 결과 |
| `academy_unassigned_by_gu.csv` | 구별 학원 동 미배정 비율 |

## transactions.parquet 컬럼
| 컬럼 | 설명 |
|---|---|
| row_id | 원자료 행 순서(0부터). 고유키 |
| src_no | 원자료 NO(다운로드 묶음마다 1부터 다시 시작, 고유키 아님) |
| gu, dong, dong_id | 구, 법정동, "<구> <법정동>" 조인 키 |
| dong_key_missing | 시군구에서 구·동을 읽지 못한 행(결과 {fc['dong_key_missing']}행) |
| complex_id | 단지 키 = 구\\|법정동\\|단지명\\|번지 |
| complex_name, jibun, road_name | 단지명, 번지, 도로명 |
| area_m2, floor, build_year | 전용면적(㎡), 층, 건축년도 |
| price_manwon, price_eok | 거래금액(만원), 거래금액(억원) |
| ppa_manwon, log_ppa | ㎡당 가격(만원/㎡), 그 자연로그 |
| contract_date, contract_ym, contract_year | 계약일, 계약월(YYYY-MM), 계약연도 |
| age | 계약연도 − 건축년도(음수 가능) |
| area_bin, age_bin, floor_bin | 분석용 구간(기준 범주: 060-085, 10-14, 06-10) |
| deal_type | 거래유형(중개/직거래). 2021-11 이후만 기록 |
| cancel_date, registration_date | 해제사유발생일, 등기일자(원문) |
| is_cancelled | 취소(해제) 거래 ({fc['is_cancelled']:,}행) |
| cancel_unobserved | 계약월 < 2020-02: 취소 정보 자체가 없는 구간 ({fc['cancel_unobserved']:,}행) |
| is_dup_exact | NO를 뺀 20개 열이 앞 행과 모두 같은 완전 중복(두 번째부터 표시, {fc['is_dup_exact']:,}행) |
| bulk_n | 같은 단지·같은 계약일의 유효(취소·중복 제외) 거래 수 |
| is_bulk | 유효 거래 & bulk_n ≥ {BULK_N} (일괄 거래, {fc['is_bulk']:,}행). 기준 근거는 노트북 2-3절 |
| is_partial_month | 2017-10(10-28부터), 2026-10(10-26까지) 부분 월 |
| is_recent_provisional | 2026-08, 2026-09: 취소·신고 지연이 끝나지 않은 잠정 월 |
| is_floor_nonpositive, is_age_negative, is_small_unit | 층 ≤ 0, 연식 < 0, 전용 20㎡ 미만 |
| price_robust_z, is_price_outlier | 구·연도 안 log(㎡당 가격) 로버스트 z, \\|z\\|>5 |
| is_direct_trade | 직거래 |
| in_base | 기본 표본: 취소 아님·완전 중복 아님·부분 월 아님·동 키 있음 ({fc['in_base']:,}행, 2017-11~2026-09) |
| in_base_nobulk | in_base & 일괄 거래 아님 ({fc['in_base_nobulk']:,}행) |

## dong_features.parquet 컬럼
| 컬럼 | 설명 | 시점 |
|---|---|---|
| gu, dong, dong_id | 키 | - |
| lat, lon, coord_source, coord_n_points, coord_missing | 동 대표 좌표(단지 좌표 중앙값, 없으면 학교 좌표 중앙값) | 현재 단지 목록 |
| n_school_elem/middle/high/total | 초·중·고 학교 수(실제 0 포함) | **현재 시점** {SCHOOL_ASOF} |
| acad_exam | 학원·교습소 중 분야 '입시.검정 및 보습' 수 | **현재 시점** |
| acad_exam_hagwon | acad_exam 중 '학원'만 | **현재 시점** |
| acad_lang / acad_arts / acad_other / acad_total | '국제화' / '예능(대)' / 그 밖의 분야 / 합계 | **현재 시점** |
| commute_gwanghwamun/gangnam/yeouido | 대중교통 통근시간(분), 구글 경로 | **현재 시점**, 출발 {'; '.join(cm_dep)} |
| commute_min, commute_min_dest | 3곳 중 최솟값(분)과 해당 업무지구 | **현재 시점** |
| commute_missing | 통근 CSV에 없는 동(결측, 대체하지 않음) | - |
| subway_dist_m | 최근접 지하철역 거리(m, 단지별 값의 동 중앙값) | **현재 역 목록** |
| subway_n_1km | 반경 1km 역 수(동 중앙값) | **현재 역 목록** |
| nearest_transfer_share | 최근접 역이 환승역인 단지 비율 | **현재 역 목록** |
| bus_n_500m | 반경 500m 버스정류소 수(동 중앙값, 한강버스 제외) | **현재 정류소 목록** |
| gu_businesses_2024, gu_workers_2024 | **구 단위** 사업체 수·종사자 수(2024) | 2024년 |
| apt_n_complex, apt_households_median, apt_households_total, apt_large_share, apt_info_missing | 단지 수, 세대수 중앙값·합, 1,000세대 이상 단지 비율, 단지정보 결측 | **현재 단지 목록** |
| median_build_year, tx_n_base | 기본 표본 거래의 건축년도 중앙값, 거래 수 | 2017-11~2026-09 전체 |

## 인계 규칙(ml-engineer 포함)
1. 모든 조인은 `dong_id`로 한다. 동 이름만으로 조인하지 않는다.
2. `dong_features_meta.csv`에서 `usable_in_backtest=false`인 특징은 **현재 시점 횡단면 분석과 04·05의 연관 분석에만** 쓴다. **07-1 주 백테스트(워크포워드)에는 넣지 않는다.** 현재 `dong_features`의 모든 특징(동 대표 좌표 `lat`·`lon` 포함)이 false다. 좌표도 현재 단지·학교 목록으로 계산했기 때문이다.
3. 통근시간 결측 동은 대체하지 않는다. 쓰려면 결측 지시 변수(`commute_missing`)와 함께 쓰고, 결과에 결측 동 수를 적는다.
4. 학원 수 0은 "배정된 학원이 없음"이다. 괄호 주소가 없는 학원은 어느 동에도 배정되지 않았다.
5. 사업체·종사자는 구 단위 값이라 구 고정효과와 함께 쓰면 식별되지 않는다.
6. 기본 분석 표본은 `in_base`, 횡단면·패널 주결과는 `in_base_nobulk`를 권장한다. 2026-08~09는 잠정 월, 2020-02 이전은 취소 정보 없음.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
len(readme)

3920

## 결론
- 정제 거래 자료 `transactions.parquet`는 원자료와 같은 547,735행이며, 삭제 없이 플래그로 표시했다(플래그 행 수는 3절 표, `flag_counts.csv`).
- 일괄 거래 기준은 같은 단지·같은 날 유효 거래 10건 이상이다. 근거는 2-3절의 묶음 크기별 빈도 감소비다.
- 동 특징표 `dong_features.parquet`는 338개 법정동, 키는 (`gu`, `dong`)와 `dong_id`이며 중복이 없다(5-7절).
- 통근시간은 API로 새로 수집하지 않았고, CSV에 없는 동은 결측으로 남겼다(5-3절, `commute_missing_dongs.csv`).

## 한계
- 2020-02 이전 계약은 취소 정보가 없어 취소 거래가 섞여 있을 수 있다.
- 2026-08~09는 취소·신고 지연이 끝나지 않은 잠정 월이다.
- 교육·통근·지하철·버스·단지 정보는 모두 현재 시점 자료여서 과거 거래와 시점이 맞지 않는다(`dong_features_meta.csv`).
- 사업체·종사자는 행정동-법정동 대응표가 원자료에 없어 구 단위로만 붙였다.
- 통근시간의 '강남'은 파일에 정해진 한 지점 기준이어서 삼성·대치 등 같은 업무권역 안에서도 값이 크게 다를 수 있다.
- 학원 동 배정은 도로명상세주소 괄호 표기에 의존하므로 일부 학원이 빠졌다.